# Week08 Notebook

## Intro on Moodle

This project will focus on making recommendations of posts to redditors based on their voting history. The project will use the following dataset, Huge Collection of Reddit Votes | Kaggle, which contains over 22m posts and 44m votes spanning 139k subreddits. But don’t be intimidated by the number, after some preliminary analysis, only 356 subreddits have more than 50 users who cast more than 50 votes. But they are not the only possible target audience for this project, it could study the effectiveness of various recommender system on users who cast few votes. If you are so keen, you can try your hands on Large Vision and Language Model, given r/meme and r/darkmeme are among the most active subreddits.


## 1. Define the Problem Scope
Before diving into the data, your team needs to precisely define the recommendation problem you are solving. A good starting point is to ask:

Who are you recommending to? Are you targeting new users with little to no voting history (the "cold start" problem) or established users with extensive data? Your preliminary analysis about users with few vs. many votes is a great starting point for this.


What are you recommending? Are you recommending individual posts from subreddits the user is already subscribed to, or posts from new, undiscovered subreddits? 


How will it work? Define the user inputs (e.g., upvotes, downvotes, comments), what the recommendation output will look like (e.g., a ranked list of 10 posts), and how the system might gather feedback. You could even create simple mockups of a user interface to illustrate this.


How does it compare? Briefly analyze Reddit's current recommendation system. What are its strengths and what limitations could your project address? This "competitor analysis" is a required part of the report.

Yes, using a Large Language Model (LLM) is an excellent idea for an advanced technique in your project. It aligns perfectly with the requirement to explore more complex methods and can be a major point of discussion in your report and presentation.


Here are two main ways you can implement an LLM approach for your Reddit post recommender.

### LLM as a Feature Encoder
This is the most direct and practical method. Instead of using simpler techniques like TF-IDF to understand the content of a post, you use a powerful LLM to generate rich numerical representations (embeddings) of the text.

How it works:

Generate Embeddings: For each post, you take its title and/or self-text and feed it into a pre-trained LLM (like models from the BERT family or Sentence-Transformers). The model outputs a dense vector (an embedding) that captures the semantic meaning of that text. For meme-based subreddits, you could use a multimodal model (like CLIP) to create embeddings that represent both the image and the text.

Use the Embeddings: Once you have an embedding for every post, you can use them in two primary ways:

In a Content-Based System: You can build a user profile by averaging the embeddings of all the posts they have upvoted. To find recommendations, you calculate the cosine similarity between this user profile vector and the vectors of all candidate posts, recommending the ones with the highest similarity.

In a Hybrid System: These rich content embeddings can be used as features in a more complex model. For example, you could build a neural network that takes both a user's ID (as an embedding) and a post's LLM-generated content embedding as input to predict the probability of an upvote.

This approach significantly enhances the "content understanding" part of your recommender.

In [1]:
import kagglehub
import os
import pandas as pd

c:\Users\z5612172\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def download_dataset()->list[str]:
    """
    Download the dataset from Kaggle and return the paths to the files.
    """
    dataset_dir = kagglehub.dataset_download("josephleake/huge-collection-of-reddit-votes")
    paths = []
    for dir_path, _, file_names in os.walk(dataset_dir):
        for file_name in file_names:
            paths.append(os.path.join(dir_path, file_name))
    print(f'File path to votes:\n{paths[0]}')
    print(f'File path to submissions:\n{paths[1]}')
    return paths

def get_dataframe(sample=None)->tuple[pd.DataFrame]:
    """
    Return a tuple of two pandas.Dataframe: votes and submissions.

    Returns:
        tuple[pd.DataFrame]: a tuple of two dataframes.
    """
    paths = download_dataset()
    if sample is None:
        votes = pd.read_csv(paths[0], sep='\t')
        submissions = pd.read_csv(paths[1], sep='\t')
    else:
        votes = pd.read_csv(paths[0], sep='\t').sample(sample, random_state=42)
        submissions = pd.read_csv(paths[1], sep='\t').sample(sample, random_state=42)
    return (votes, submissions)

def view_users_votes(votes:pd.DataFrame):
    view = (
        votes
        .groupby(['USERNAME', 'SUBREDDIT', 'VOTE'])
        .size()                         # count upvotes/downvotes in each group
        .unstack(fill_value=0)          # pivot VOTE labels into columns
        .rename(columns={
            'upvote':   'num_upvotes',
            'downvote': 'num_downvotes'
        })
        .reset_index()                  # turn USERNAME & SUBREDDIT back into columns
    )
    return view

In [10]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

# --- 1. Load Data ---
# It's critical to start with a small sample.
# Load a random sample of 50,000 posts from the dataset.
# Make sure the 'posts.csv' file from the Kaggle dataset is in the same directory.
# try:
#     posts_df = pd.read_csv('posts.csv').sample(n=50000, random_state=42)
# except FileNotFoundError:
#     print("Error: 'posts.csv' not found. Please download it from the Kaggle dataset.")
#     exit()

votes, posts = get_dataframe(sample=25000)


print(f"Loaded {len(posts)} sample submissions.")
# We only need the ID and title for this task.
posts=posts.rename(columns={'SUBMISSION_ID':"id",'TITLE':'title','SUBREDDIT':'subreddit','AUTHOR':'author'})
posts = posts[['id', 'title']].dropna(subset=['title'])

print(f"Processing {len(posts)} posts after dropping ones with no title.")


# --- 2. Initialize Model and Generate Embeddings ---
# Load a pre-trained model from sentence-transformers.
# 'all-MiniLM-L6-v2' is a good, fast model for general purpose use.
print("Loading the sentence transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Prepare the sentences for the model.
# The model expects a list of strings.
post_titles = posts['title'].tolist()

print("Generating embeddings for post titles... (This may take a few minutes)")
# The encode() method takes a list of strings and returns a list of embeddings.
# We specify `show_progress_bar=True` to see the progress.
post_embeddings = model.encode(post_titles, show_progress_bar=True)

print("Embeddings generated successfully!")
print(f"Shape of the embeddings array: {post_embeddings.shape}") # (num_posts, embedding_dimension)


# --- 3. Save the Results ---
# It's essential to save your embeddings so you don't have to regenerate them.
# We'll save the post IDs and their corresponding embeddings.
post_ids = posts['id'].tolist()

# Use numpy's savez_compressed to save multiple arrays efficiently.
np.savez_compressed(
    'post_embeddings.npz',
    ids=post_ids,
    embeddings=post_embeddings
)

print("\nSuccessfully saved post IDs and embeddings to 'post_embeddings.npz'.")
print("You can now load this file in another script to build your recommender.")

File path to votes:
C:\Users\z5612172\.cache\kagglehub\datasets\josephleake\huge-collection-of-reddit-votes\versions\1\44_million_reddit_votes\44_million_votes.txt
File path to submissions:
C:\Users\z5612172\.cache\kagglehub\datasets\josephleake\huge-collection-of-reddit-votes\versions\1\submission_info\submission_info.txt
Loaded 25000 sample submissions.
Processing 25000 posts after dropping ones with no title.
Loading the sentence transformer model...
Generating embeddings for post titles... (This may take a few minutes)


Batches: 100%|██████████| 782/782 [00:35<00:00, 22.12it/s]


Embeddings generated successfully!
Shape of the embeddings array: (25000, 384)

Successfully saved post IDs and embeddings to 'post_embeddings.npz'.
You can now load this file in another script to build your recommender.


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import os

# --- Mock Data Setup ---
# In your actual project, you would load your 'submissions' dataframe here.
# For this demonstration, we'll create a mock dataframe that mirrors your data structure.
# print("--- 1. Setting up Mock Data ---")
# mock_data = {
#     'SUBMISSION_ID': [f't3_{i}' for i in range(10)],
#     'SUBREDDIT': [
#         'r/science', 'r/memes', 'r/python', 'r/science', 'r/AskReddit',
#         'r/memes', 'r/gaming', 'r/python', 'r/science', 'r/memes'
#     ],
#     'TITLE': ['A new discovery' for _ in range(10)]
# }
# submissions_df = pd.DataFrame(mock_data)
# print("Sample Submissions DataFrame:")
# print(submissions_df.head())
# print("-" * 30)


# --- 2. Prepare Subreddit Data for Embedding ---
# An embedding layer needs integer indices as input, not strings.
# So, we create a mapping from each unique subreddit name to a unique integer.
print("\n--- 2. Preparing Subreddit Data ---")
unique_subreddits = posts['subreddit'].unique().tolist()
subreddit_to_idx = {name: i for i, name in enumerate(unique_subreddits)}
idx_to_subreddit = {i: name for i, name in enumerate(unique_subreddits)}

# Add the integer index to our dataframe
posts['subreddit_idx'] = posts['subreddit'].map(subreddit_to_idx)
posts.head(10)

print(f"Found {len(unique_subreddits)} unique subreddits.")
print("Mapping from subreddit name to index:")
print(subreddit_to_idx)
print("\nDataFrame with subreddit indices:")
print(posts.head())
print("-" * 30)


# --- 3. Define and Initialize the Embedding Layer ---
# We'll use a simple PyTorch model to house our embedding layer.
# print("\n--- 3. Defining the PyTorch Model ---")

# # Parameters for our embedding layer
# num_subreddits = len(unique_subreddits)
# embedding_dim = 10 # This is a hyperparameter you can tune. It's the size of the vector for each subreddit.

# class SubredditEmbedder(nn.Module):
#     def __init__(self, num_categories, emb_dimension):
#         super(SubredditEmbedder, self).__init__()
#         # The embedding layer:
#         # It's essentially a lookup table where each row is a vector.
#         # The first argument is the number of unique categories (our subreddits).
#         # The second argument is the desired size of the embedding vector for each category.
#         self.embedding_layer = nn.Embedding(num_categories, emb_dimension)

#     def forward(self, category_indices):
#         # The forward pass takes a tensor of indices and returns the corresponding embeddings.
#         return self.embedding_layer(category_indices)

# # Instantiate the model
# model = SubredditEmbedder(num_subreddits, embedding_dim)
# print(f"Successfully created a model with an embedding layer.")
# print(f"The embedding layer will map each of the {num_subreddits} subreddits to a vector of size {embedding_dim}.")
# print("-" * 30)


# # --- 4. Get an Embedding for a Specific Subreddit ---
# print("\n--- 4. Getting an Embedding ---")
# # Let's get the embedding for 'r/python'
# subreddit_name = 'r/python'
# subreddit_idx = subreddit_to_idx[subreddit_name]

# # Convert the index to a PyTorch tensor. The model expects a tensor as input.
# # The tensor should be of type Long (64-bit integer).
# subreddit_tensor = torch.LongTensor([subreddit_idx])

# # Get the embedding vector from the model
# # `torch.no_grad()` is used as we are not training, just doing inference.
# with torch.no_grad():
#     embedding_vector = model(subreddit_tensor)

# print(f"The learned embedding vector for '{subreddit_name}' (index {subreddit_idx}) is:")
# print(embedding_vector)
# print(f"Vector shape: {embedding_vector.shape}")
# print("-" * 30)

# # --- 5. How to Use This in Your Project ---
# print("\n--- 5. Next Steps ---")
# print("1. Train the Model: In a real recommender system, this embedding layer would be part of a larger model.")
# print("   You would train the full model on a task (e.g., predicting user upvotes), and the embedding vectors would be learned automatically during training.")
# print("2. Concatenate Features: Once trained, you can get the learned subreddit embedding for a post and concatenate it with the post's title embedding (from the SentenceTransformer).")
# print("   This creates a rich, combined feature vector for each post.")